In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/tez_transformer_v4_repro")
ROBUST = ROOT / "reproducibility" / "bartlett_dm_hln_v1"

kontroller = [
    ROOT,
    ROOT / "scripts",
    ROOT / "results",
    ROOT / "results/statistical_comparison/09_dm_harvey_holm_results_v4.csv",
    ROOT / "results/baselines/naive/naive_baseline_loss_series_v4.npz",
    ROOT / "results/baselines/learned/learned_baseline_loss_series_v4.npz",
    ROOT / "results/baselines/garch",
    ROBUST,
]

for yol in kontroller:
    print("✅" if yol.exists() else "❌", yol)

print("\nYüklenen ZIP dosyaları:")
zipler = sorted(ROBUST.glob("*.zip"))

for dosya in zipler:
    print(f"✅ {dosya.name} — {dosya.stat().st_size / 1024**2:.2f} MB")

print(f"\nToplam ZIP sayısı: {len(zipler)}")

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

print("Proje klasöründe bulunan ZIP dosyaları:\n")

zipler = sorted(ROOT.rglob("*.zip"))

if zipler:
    for dosya in zipler:
        print("✅", dosya)
else:
    print("❌ Proje klasöründe ZIP bulunamadı.")

In [ ]:
from pathlib import Path
import hashlib

ROOT = Path("/content/drive/MyDrive/tez_transformer_v4_repro")
ROBUST = ROOT / "reproducibility bartlett_dm_hln_v1"

beklenen = {
    "drive-download-20260909T063323Z-1-001.zip":
        "bc7dac4b88d8808fe556351a7045b524832ab8588148736243c230370ad56871",

    "Bartlett_DM_HLN_Denetim_Paketi_v1.zip":
        "e7aba2784d2710ff27262c430e1b7614ac7d7f3700551f77fd097ec2cc338e9e",
}

def sha256_hesapla(dosya):
    h = hashlib.sha256()
    with open(dosya, "rb") as f:
        for parca in iter(lambda: f.read(1024 * 1024), b""):
            h.update(parca)
    return h.hexdigest()

for ad, beklenen_hash in beklenen.items():
    dosya = ROBUST / ad
    hesaplanan = sha256_hesapla(dosya)

    if hesaplanan == beklenen_hash:
        print("✅ PASS:", ad)
    else:
        print("❌ HASH UYUŞMUYOR:", ad)
        print("Hesaplanan:", hesaplanan)

In [ ]:
from pathlib import Path
import tempfile
import zipfile

ROOT = Path("/content/drive/MyDrive/tez_transformer_v4_repro")
ROBUST = ROOT / "reproducibility bartlett_dm_hln_v1"

SOURCE_ZIP = ROBUST / "drive-download-20260909T063323Z-1-001.zip"
AUDIT_ZIP = ROBUST / "Bartlett_DM_HLN_Denetim_Paketi_v1.zip"

# Her çalıştırmada temiz ve geçici bir klasör oluşturur.
WORK = Path(tempfile.mkdtemp(prefix="bartlett_repro_v1_"))
SOURCE_ROOT = WORK / "source"
AUDIT_ROOT = WORK / "audit"
OUTPUT_ROOT = WORK / "output"

SOURCE_ROOT.mkdir()
AUDIT_ROOT.mkdir()
OUTPUT_ROOT.mkdir()

with zipfile.ZipFile(SOURCE_ZIP, "r") as z:
    z.extractall(SOURCE_ROOT)

with zipfile.ZipFile(AUDIT_ZIP, "r") as z:
    z.extractall(AUDIT_ROOT)

kontroller = [
    SOURCE_ROOT / "scripts",
    SOURCE_ROOT / "results",
    SOURCE_ROOT / "results/statistical_comparison/09_dm_harvey_holm_results_v4.csv",
    SOURCE_ROOT / "results/baselines/naive/naive_baseline_loss_series_v4.npz",
    SOURCE_ROOT / "results/baselines/learned/learned_baseline_loss_series_v4.npz",
    SOURCE_ROOT / "results/baselines/garch",
    AUDIT_ROOT / "bartlett_dm_hln_robustness_v1.py",
    AUDIT_ROOT / "protokol_bartlett_dm_hln_v1.md",
]

for yol in kontroller:
    print("✅" if yol.exists() else "❌", yol)

print("\nGeçici çalışma klasörü:", WORK)

In [ ]:
import subprocess
import sys
import json
import numpy as np
import pandas as pd

# Analizde kullanılan doğrulama kütüphanesi
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "statsmodels==0.14.5"],
    check=True
)

ANALIZ_KODU = AUDIT_ROOT / "bartlett_dm_hln_robustness_v1.py"
PROTOKOL = AUDIT_ROOT / "protokol_bartlett_dm_hln_v1.md"

komut = [
    sys.executable,
    str(ANALIZ_KODU),
    "--project-root", str(SOURCE_ROOT),
    "--output-dir", str(OUTPUT_ROOT),
    "--protocol", str(PROTOKOL),
    "--source-zip", str(SOURCE_ZIP),
]

print("Bartlett-HAC analizi çalıştırılıyor...\n")
subprocess.run(komut, check=True)

# Üretilen sonuç özetini oku
with open(
    OUTPUT_ROOT / "bartlett_dm_hln_summary_v1.json",
    encoding="utf-8"
) as f:
    ozet = json.load(f)

# Benim hazırladığım sonuçlarla Colab sonuçlarını karşılaştır
referans = pd.read_csv(
    AUDIT_ROOT / "bartlett_dm_hln_results_v1.csv"
)
colab_sonucu = pd.read_csv(
    OUTPUT_ROOT / "bartlett_dm_hln_results_v1.csv"
)

sayisal = referans.select_dtypes(include=np.number).columns
maksimum_fark = (
    referans[sayisal] - colab_sonucu[sayisal]
).abs().to_numpy().max()

diger = [sutun for sutun in referans.columns if sutun not in sayisal]

kararlar_ayni = (
    referans[diger].astype(str).reset_index(drop=True).equals(
        colab_sonucu[diger].astype(str).reset_index(drop=True)
    )
)

assert ozet["validation"]["status"] == "PASS"
assert ozet["comparison_count"] == 22
assert ozet["changed_decision_count"] == 6
assert ozet["direction_reversal_count"] == 0
assert ozet["bartlett_counts"] == {
    "SIGNIFICANT_FINAL_BETTER": 2,
    "SIGNIFICANT_BASELINE_BETTER": 12,
    "NOT_SIGNIFICANT": 8,
}
assert maksimum_fark < 1e-10
assert kararlar_ayni

print("\n✅ COLAB YENİDEN ÜRETİMİ: PASS")
print("Karşılaştırma sayısı:", ozet["comparison_count"])
print("Nihai model üstün:", ozet["bartlett_counts"]["SIGNIFICANT_FINAL_BETTER"])
print("Kıyaslama modeli üstün:", ozet["bartlett_counts"]["SIGNIFICANT_BASELINE_BETTER"])
print("Anlamlı fark yok:", ozet["bartlett_counts"]["NOT_SIGNIFICANT"])
print("Değişen karar:", ozet["changed_decision_count"])
print("Yön değişikliği:", ozet["direction_reversal_count"])
print("Referansla en büyük sayısal fark:", maksimum_fark)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import shutil
import scipy
import statsmodels

ROOT = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

ESKI_KLASOR = ROOT / "reproducibility bartlett_dm_hln_v1"
REPRO_DIR = ROOT / "reproducibility" / "bartlett_dm_hln_v1"
RESULT_DIR = ROOT / "results" / "robustness" / "bartlett_dm_hln_v1"
SCRIPT_DIR = ROOT / "scripts"

REPRO_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def sha256(dosya):
    h = hashlib.sha256()
    with open(dosya, "rb") as f:
        for parca in iter(lambda: f.read(1024 * 1024), b""):
            h.update(parca)
    return h.hexdigest()

def kopyala_ve_dogrula(kaynak, hedef):
    kaynak = Path(kaynak)
    hedef = Path(hedef)
    shutil.copy2(kaynak, hedef)

    if sha256(kaynak) != sha256(hedef):
        raise RuntimeError(f"Kopyalama doğrulanamadı: {hedef}")

    print("✅", hedef.relative_to(ROOT))

# İki ZIP'i düzgün klasör yapısına kopyala
for ad in [
    "drive-download-20260909T063323Z-1-001.zip",
    "Bartlett_DM_HLN_Denetim_Paketi_v1.zip",
]:
    kopyala_ve_dogrula(ESKI_KLASOR / ad, REPRO_DIR / ad)

# Colab yeniden üretim kaydı
yeniden_uretim_kaydi = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS",
    "comparison_count": 22,
    "bartlett_counts": {
        "final_model_better": 2,
        "baseline_model_better": 12,
        "not_significant": 8,
    },
    "changed_decision_count": 6,
    "direction_reversal_count": 0,
    "max_abs_numeric_difference_from_reference": float(maksimum_fark),
    "comparison_tolerance": 1e-10,
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "statsmodels": statsmodels.__version__,
    },
}

kayit_dosyasi = OUTPUT_ROOT / "colab_reproduction_check_v1.json"
kayit_dosyasi.write_text(
    json.dumps(yeniden_uretim_kaydi, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

# Colab'ın yeniden ürettiği bütün sonuçları kaydet
for dosya in OUTPUT_ROOT.iterdir():
    if dosya.is_file():
        kopyala_ve_dogrula(dosya, RESULT_DIR / dosya.name)

# Dondurulmuş protokol ve çalışma kitabı
for ad in [
    "protokol_bartlett_dm_hln_v1.md",
    "Bartlett_DM_HLN_Saglamlik_Analizi_v1.xlsx",
]:
    kopyala_ve_dogrula(AUDIT_ROOT / ad, RESULT_DIR / ad)

# Analiz kodunu ana scripts klasörüne kaydet
kopyala_ve_dogrula(
    AUDIT_ROOT / "bartlett_dm_hln_robustness_v1.py",
    SCRIPT_DIR / "09D_bartlett_dm_hln_robustness_v1.py",
)

print("\n✅ BARTLETT ANALİZİ DRIVE'A KAYDEDİLDİ")
print("Kod:", SCRIPT_DIR / "09D_bartlett_dm_hln_robustness_v1.py")
print("Sonuçlar:", RESULT_DIR)
print("Arşivler:", REPRO_DIR)

In [ ]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/tez_transformer_v4_repro")
REPRO_DIR = ROOT / "reproducibility" / "bartlett_dm_hln_v1"
RESULT_DIR = ROOT / "results" / "robustness" / "bartlett_dm_hln_v1"
SCRIPT = ROOT / "scripts" / "09D_bartlett_dm_hln_robustness_v1.py"

beklenen_sonuclar = [
    "colab_reproduction_check_v1.json",
    "bartlett_loss_differentials_v1.npz",
    "bartlett_dm_hln_autocovariances_v1.csv",
    "bartlett_dm_hln_manifest_v1.json",
    "bartlett_dm_hln_summary_v1.json",
    "bartlett_dm_hln_changed_decisions_v1.csv",
    "bartlett_dm_hln_report_v1.md",
    "bartlett_loss_differentials_long_v1.csv",
    "bartlett_dm_hln_results_v1.csv",
    "protokol_bartlett_dm_hln_v1.md",
    "Bartlett_DM_HLN_Saglamlik_Analizi_v1.xlsx",
]

def sha256(dosya):
    h = hashlib.sha256()
    with open(dosya, "rb") as f:
        for parca in iter(lambda: f.read(1024 * 1024), b""):
            h.update(parca)
    return h.hexdigest()

for ad in beklenen_sonuclar:
    dosya = RESULT_DIR / ad
    assert dosya.exists(), f"Eksik dosya: {dosya}"

assert SCRIPT.exists()

assert sha256(
    RESULT_DIR / "protokol_bartlett_dm_hln_v1.md"
) == "70a7aeb6bbfb0dfcfe7375e2d4e678837762af3b720e5b08c00fd8110f0074fc"

assert sha256(
    REPRO_DIR / "drive-download-20260909T063323Z-1-001.zip"
) == "bc7dac4b88d8808fe556351a7045b524832ab8588148736243c230370ad56871"

assert sha256(
    REPRO_DIR / "Bartlett_DM_HLN_Denetim_Paketi_v1.zip"
) == "e7aba2784d2710ff27262c430e1b7614ac7d7f3700551f77fd097ec2cc338e9e"

with open(
    RESULT_DIR / "colab_reproduction_check_v1.json",
    encoding="utf-8"
) as f:
    kontrol = json.load(f)

assert kontrol["status"] == "PASS"
assert kontrol["comparison_count"] == 22
assert kontrol["changed_decision_count"] == 6

print("✅ DRIVE NİHAİ KONTROLÜ: PASS")
print("Sonuç dosyası:", len(beklenen_sonuclar))
print("Analiz kodu: mevcut")
print("Protokol hash: doğrulandı")
print("Kaynak ZIP hash: doğrulandı")
print("Denetim paketi hash: doğrulandı")